In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import sys
import os
import csv
from sklearn.linear_model import LinearRegression
import requests
import plotly
import math
import openpyxl as xl

import xlwings as xw
import plotly.express as px
import xlsxwriter
from openpyxl import Workbook
from openpyxl.drawing.image import Image


import bs4 as bs
import time

import pandas_datareader as web
##import pandas_datareader.data as web
import pickle
import requests
import json
import yfinance as yf

import yfinance as yf
import pandas as pd
import time
import yfinance as yf

from datetime import datetime, timedelta
from iexfinance.stocks import Stock
from datetime import datetime, timedelta
from iexfinance.stocks import get_historical_data
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import scipy.stats as stats

In [8]:
import os
import time
import requests
import pandas as pd

# ===========================
# Configuration
# ===========================
BASE_PATH = '/Users/andras/OneDrive/Fundamentals/MoreComp'
OUTPUT_PATH = os.path.join(BASE_PATH, 'Comp')
COMP_PATH = os.path.join(BASE_PATH, 'CD')
TICKERS_FILE = os.path.join(BASE_PATH, 'tickers.txt')

API_KEY = '5VJ5OHNNB582TWUU'  # your Alpha Vantage API key

STATEMENTS = {
    'INCOME_STATEMENT': 'Income Statement',
    'BALANCE_SHEET': 'Balance Sheet',
    'CASH_FLOW': 'Cash Flow Statement'
}

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(COMP_PATH, exist_ok=True)

# ===========================
# Helper functions
# ===========================
def get_financial_data(symbol, function, retries=5, sleep_time=20):
    """Fetch financial data from Alpha Vantage with retry on rate-limit or empty response."""
    url = "https://www.alphavantage.co/query"
    params = {"function": function, "symbol": symbol, "apikey": API_KEY}

    for attempt in range(1, retries + 1):
        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"{symbol}: HTTP error {response.status_code} on {function}")
            time.sleep(sleep_time)
            continue

        data = response.json()

        # Handle rate-limiting
        if "Note" in data or "Information" in data:
            print(f"{symbol}: ⏳ Rate limit hit on {function}, waiting {sleep_time}s... (Attempt {attempt})")
            time.sleep(sleep_time)
            continue

        # Check for valid quarterlyReports
        if "quarterlyReports" in data:
            return data["quarterlyReports"][:20]

        print(f"{symbol}: ⚠️ No quarterlyReports found for {function}. Retrying... (Attempt {attempt})")
        time.sleep(sleep_time)

    print(f"{symbol}: ❌ Failed to retrieve {function} after {retries} retries.")
    return []

def create_dataframe(reports, statement_name):
    """Convert JSON report list to DataFrame."""
    df = pd.DataFrame()
    for report in reports:
        fiscal_date = pd.to_datetime(report.get('fiscalDateEnding')).strftime('%Y-%m-%d')
        df_quarter = pd.DataFrame.from_dict(report, orient='index')
        df_quarter.columns = [fiscal_date]
        df_quarter.index.name = statement_name
        df = pd.concat([df, df_quarter], axis=1)
    return df

def clean_dynamic_df(df):
    """Clean combined DataFrame into a readable, dynamic format."""
    df.columns = [str(c).split(".")[0] for c in df.columns]
    df = df.groupby(df.columns, axis=1).first()

    unnamed_cols = [c for c in df.columns if 'Unnamed' in c]
    if unnamed_cols:
        df = pd.concat([df[unnamed_cols], df.drop(columns=unnamed_cols)], axis=1)
        df.rename(columns={unnamed_cols[0]: 'Line Item'}, inplace=True)
    else:
        df.reset_index(inplace=True)
        df.rename(columns={'index': 'Line Item'}, inplace=True)

    if 'Line Item' in df.columns:
        df = df[df['Line Item'] != 'fiscalDateEnding']

    data = {
        'Line Item': df.iloc[:, 0].tolist(),
        **df.iloc[:, 1:].sort_index(axis=1, ascending=False).to_dict(orient='list')
    }
    df_dynamic = pd.DataFrame(data)
    df_dynamic.set_index('Line Item', inplace=True)
    return df_dynamic

def get_company_overview(symbol, retries=5, sleep_time=12):
    """Fetch company overview from Alpha Vantage."""
    url = 'https://www.alphavantage.co/query'
    params = {'function': 'OVERVIEW', 'symbol': symbol, 'apikey': API_KEY}

    for attempt in range(1, retries + 1):
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"{symbol}: HTTP error {response.status_code} on OVERVIEW")
            time.sleep(sleep_time)
            continue

        data = response.json()
        if "Note" in data or "Information" in data:
            print(f"{symbol}: ⏳ Rate limit hit on OVERVIEW, waiting {sleep_time}s... (Attempt {attempt})")
            time.sleep(sleep_time)
            continue

        if data:
            df = pd.DataFrame.from_dict(data, orient='index', columns=['Value'])
            df.index.name = 'Attribute'
            return df

        print(f"{symbol}: ⚠️ No company overview data returned. Retrying... (Attempt {attempt})")
        time.sleep(sleep_time)

    print(f"{symbol}: ❌ Failed to retrieve company overview after {retries} retries.")
    return None

# ===========================
# Main per-ticker workflow
# ===========================
def process_ticker(ticker):
    combined_df = pd.DataFrame()

    # Fetch all financial statements
    for func, name in STATEMENTS.items():
        reports = get_financial_data(ticker, func)
        if reports:
            df_section = create_dataframe(reports, name)
            combined_df = pd.concat([combined_df, df_section], axis=1)
            print(f"{ticker}: {name} retrieved ({len(df_section.columns)} quarters).")
        else:
            print(f"{ticker}: ⚠️ No {name} data found.")

    # Save financial data
    if not combined_df.empty:
        df_clean = clean_dynamic_df(combined_df)
        output_file = os.path.join(OUTPUT_PATH, f"df_{ticker}.xlsx")
        df_clean.to_excel(output_file, index=True)
        print(f"{ticker}: ✅ Financial Data saved to {output_file}")

    # Fetch company overview
    company_df = get_company_overview(ticker)
    if company_df is not None:
        out_file = os.path.join(COMP_PATH, f"CD_{ticker}.xlsx")
        company_df.to_excel(out_file, index=True)
        print(f"{ticker}: 🏢 Company Data saved to {out_file}")

# ===========================
# Batch Runner
# ===========================
if __name__ == "__main__":
    with open(TICKERS_FILE, "r") as f:
        tickers = [line.strip().upper() for line in f if line.strip()]

    print(f"Found {len(tickers)} tickers: {', '.join(tickers)}")

    for ticker in tickers:
        print(f"\n==============================\n🚀 Processing ticker: {ticker}\n==============================")
        try:
            process_ticker(ticker)
            # Wait between tickers to avoid rate limits
            time.sleep(12)
        except Exception as e:
            print(f"❌ Error processing {ticker}: {e}")


Found 1 tickers: CVI

🚀 Processing ticker: CVI
CVI: Income Statement retrieved (20 quarters).
CVI: ⏳ Rate limit hit on BALANCE_SHEET, waiting 20s... (Attempt 1)
CVI: Balance Sheet retrieved (20 quarters).
CVI: ⏳ Rate limit hit on CASH_FLOW, waiting 20s... (Attempt 1)
CVI: Cash Flow Statement retrieved (20 quarters).
CVI: ✅ Financial Data saved to /Users/andras/OneDrive/Fundamentals/MoreComp/Comp/df_CVI.xlsx
CVI: ⏳ Rate limit hit on OVERVIEW, waiting 12s... (Attempt 1)
CVI: 🏢 Company Data saved to /Users/andras/OneDrive/Fundamentals/MoreComp/CD/CD_CVI.xlsx


In [9]:
import yfinance as yf
import pytz
from pandas.tseries.offsets import BMonthEnd

DIVIDEND_PATH = '/Users/andras/OneDrive/Fundamentals/MoreComp/Dividend'
os.makedirs(DIVIDEND_PATH, exist_ok=True)

def export_dividends(ticker):
    timezone = 'America/New_York'
    today = pd.Timestamp.now(tz=pytz.timezone(timezone)).normalize()

    series = yf.Ticker(ticker).dividends

    if not series.empty:
        series = series.sort_index(ascending=False)
        dfdi = (
            series.to_frame(name='Dividend')
            .reset_index()
            .rename(columns={'index': 'Date'})
        )
        dfdi['Date'] = pd.to_datetime(dfdi['Date']).dt.tz_localize(None)
        data = dfdi[['Date', 'Dividend']].copy()

    else:
        first_date = BMonthEnd().rollback(today)
        if first_date >= today:
            first_date = BMonthEnd().rollback(first_date - pd.DateOffset(days=1))
        dates = []
        for i in range(20):
            candidate = first_date - pd.DateOffset(months=3 * i)
            candidate_bme = BMonthEnd().rollback(candidate)
            dates.append(candidate_bme.normalize())
        data = pd.DataFrame({'Date': dates, 'Dividend': 1})

    data['Date'] = pd.to_datetime(data['Date']).dt.normalize()
    data = data.sort_values('Date', ascending=False).reset_index(drop=True)
    data['Date'] = data['Date'].dt.strftime('%Y-%m-%d')

    # SAVE INTO /Dividend SUBFOLDER
    out_file = os.path.join(DIVIDEND_PATH, f"Div_{ticker}.xlsx")
    data.to_excel(out_file, index=False)

    print(f"{ticker}: 📄 Dividend file saved to {out_file}")
    return data
#Save
for ticker in tickers:
    print(f"\n==============================\n📈 Running for ticker: {ticker}\n==============================")

    try:
        export_dividends(ticker)
    except Exception as e:
        print(f"❌ Error processing {ticker}: {e}")


📈 Running for ticker: CVI
CVI: 📄 Dividend file saved to /Users/andras/OneDrive/Fundamentals/MoreComp/Dividend/Div_CVI.xlsx


In [10]:
import requests
import pandas as pd
import yfinance as yf

# Wikipedia oldal lekérése user-agent-tel
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

# HTML táblázat olvasása pandas-szal
sp500 = pd.read_html(response.text)[0]

# Nem tech szektor
non_tech = sp500[sp500['GICS Sector'] != 'Information Technology']

# Árfolyamadatok lekérése
data = []
for ticker, name in zip(non_tech['Symbol'], non_tech['Security']):
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start="2025-01-01", end="2025-10-21")
        if hist.empty:
            continue
        price_start = hist['Close'].iloc[0]
        price_end = hist['Close'].iloc[-1]
        change = price_end - price_start
        data.append([ticker, name, price_start, price_end, change])
    except Exception as e:
        print(f"Hiba a {ticker}-nél: {e}")

df = pd.DataFrame(data, columns=['Ticker', 'Company', 'Price_2025-01-01', 'Price_2026-01-31', 'Change'])
print(df)

$BRK.B: possibly delisted; no timezone found
$BF.B: possibly delisted; no price data found  (1d 2025-01-01 -> 2025-10-21)


    Ticker              Company  Price_2025-01-01  Price_2026-01-31     Change
0      MMM                   3M        126.728012        153.429306  26.701294
1      AOS          A. O. Smith         65.647514         69.091599   3.444084
2      ABT  Abbott Laboratories        110.769371        128.857269  18.087898
3     ABBV               AbbVie        171.993927        230.207794  58.213867
4      AES      AES Corporation         12.151019         14.052834   1.901814
..     ...                  ...               ...               ...        ...
425    XEL          Xcel Energy         64.215065         80.657455  16.442390
426    XYL           Xylem Inc.        114.580788        146.346100  31.765312
427    YUM          Yum! Brands        131.023422        147.461685  16.438263
428    ZBH        Zimmer Biomet        103.438240        102.467628  -0.970612
429    ZTS               Zoetis        159.795090        143.942032 -15.853058

[430 rows x 5 columns]


In [11]:
df.to_csv("non_tech_sp500_prices.csv", index=False)